## Exercise 7 - Testing Tableau

In [1]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

import shapely
from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
import folium

import os
from calitp.tables import tbl


os.environ["CALITP_BQ_MAX_BYTES"] = str(50_000_000_000)

WGS84 = "EPSG:4326"
CA_StatePlane = "EPSG:2229"  # units are in feet
CA_NAD83Albers = "EPSG:3310"  # units are in meters

SQ_MI_PER_SQ_M = 3.86 * 10 ** -7
FEET_PER_MI = 5_280
SQ_FT_PER_SQ_MI = 2.788 * 10 ** 7

GCS_FILE_PATH = "calitp-analytics-data/data-analyses/tircp"

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.10.2-CAPI-1.16.0) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(
E0321 23:09:04.594236325    1013 fork_posix.cc:70]           Fork support is only compatible with the epoll1 and poll polling strategies


#### Choose Bay Area agencies
[Agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* AC Transit: 4
* BART: 279
* SF Muni: 282 
* Samtrans: 290
*  Santa Clara Valley Transportation Authority: 294

In [14]:
#choosing a different set of ITP IDS
ITP_ID = [4,279,282,290,294]

In [15]:
bay_area = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

In [16]:
bay_area.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr
1,4,100,37.759356,-122.251237,2217 Otis Dr


In [19]:
bay_area.columns

Index(['itp_id', 'stop_id', 'stop_lat', 'stop_lon', 'stop_name'], dtype='object')

In [22]:
bay_area.sort_values(by =['itp_id','stop_id']).head(50)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr
1,4,100,37.759356,-122.251237,2217 Otis Dr
2,4,1000,37.513259,-121.943656,Auto Mall Pkwy & Osgood Rd
3,4,1001,37.513523,-121.943749,Auto Mall Pkwy & Osgood Rd
4,4,1002,37.510791,-121.954928,Auto Mall Pkwy & Technology Dr
5,4,1003,37.511324,-121.954870,Auto Mall Pkwy & Technology Dr
6,4,1004,37.484620,-121.941646,Landing Pkwy & Tennis Court
7,4,1004,37.484620,-121.941646,Landing Pkwy + Tennis Court
8,4,1005,37.482616,-121.932145,Benicia St & Auburn St
9,4,1005,37.482616,-121.932145,Benicia St + Auburn St


In [5]:
f'A total of {len(bay_area)} rows'

'A total of 27951 rows'

In [7]:
bay_area2 = bay_area.drop_duplicates(subset=['stop_id','itp_id'])

In [8]:
f'only {len(bay_area2)} rows after dropping duplicates'

'only 25010 rows after dropping duplicates'

In [9]:
#creating point geometry from longtitude &  latitude 
bay_area2 = shared_utils.geography_utils.create_point_geometry(bay_area2, 
                                                               'stop_lon', 'stop_lat')

In [10]:
bay_area2.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr,POINT (-122.25011 37.74249)
1,4,100,37.759356,-122.251237,2217 Otis Dr,POINT (-122.25124 37.75936)


In [ ]:
#bay_area2.to_file('bay_area_point.geojson', driver='GeoJSON')  

In [11]:
bay_area2.to_csv('bay_area_point.csv', index = False)  

In [ ]:
def make_routes_shapefile(ITP_ID_LIST=[], CRS="EPSG:4326", alternate_df=None):
    """
    Parameters:
    ITP_ID_LIST: list. List of ITP IDs found in agencies.yml
    CRS: str. Default is WGS84, but able to re-project to another CRS.
    Returns a geopandas.GeoDataFrame, where each line is the operator-route-line geometry.
    """

    all_routes = gpd.GeoDataFrame()

    for itp_id in ITP_ID_LIST:
        if alternate_df is None:
            shapes = (
                tbl.gtfs_schedule.shapes()
                >> filter(_.calitp_itp_id == int(itp_id))
                >> collect()
            )

        elif alternate_df is not None:
            shapes = alternate_df.copy()
            # shape_id is None, which will throw up an error later on when there's groupby
            shapes = shapes.assign(
                shape_id=shapes.route_id,
            )

        # Make a gdf
        shapes = gpd.GeoDataFrame(
            shapes,
            geometry=gpd.points_from_xy(shapes.shape_pt_lon, shapes.shape_pt_lat),
            crs=WGS84,
        )

        # Count the number of stops for a given shape_id
        # Must be > 1 (need at least 2 points to make a line)
        shapes = shapes.assign(
            num_stops=(
                shapes.groupby("shape_id")["shape_pt_sequence"].transform("count")
            )
        )

        # Drop the shape_ids that can't make a line
        shapes = shapes[shapes.num_stops > 1].reset_index(drop=True)

        # Now, combine all the stops by stop sequence, and create linestring
        unique_shapes = list(shapes.shape_id.unique())

        # Sometimes shape_id as NoneType comes up, and creates a problem, skip those
        try:
            for route in unique_shapes:
                single_shape = (
                    shapes
                    >> filter(_.shape_id == route)
                    >> mutate(shape_pt_sequence=_.shape_pt_sequence.astype(int))
                    # arrange in the order of stop sequence
                    >> arrange(_.shape_pt_sequence)
                )

            # Convert from a bunch of points to a line (for a route, there are multiple points)
            route_line = shapely.geometry.LineString(list(single_shape["geometry"]))
            single_route = single_shape[
                ["calitp_itp_id", "calitp_url_number", "shape_id"]
            ].iloc[
                [0]
            ]  # preserve info cols
            single_route["geometry"] = route_line
            single_route = gpd.GeoDataFrame(single_route, crs=WGS84)

            # https://stackoverflow.com/questions/15819050/pandas-dataframe-concat-vs-append/48168086
            all_routes = pd.concat(
                [all_routes, single_route], ignore_index=True, axis=0
            )
        except TypeError:
            print(f"unable to grab ID: {itp_id}, route: {route}")

    all_routes = (
        all_routes.to_crs(CRS)
        .sort_values(["calitp_itp_id", "shape_id"])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return all_routes


In [ ]:
bay_area3 = make_routes_shapefile(ITP_ID_LIST = [4,279,282,290,294], CRS="EPSG:4326")

In [ ]:
bay_area3 

In [ ]:
#bay_area3.to_file('bay_area_line.geojson', driver='GeoJSON')  